In [23]:
import asyncio
from collections.abc import Awaitable, Callable
from dataclasses import dataclass, field
from enum import Enum
from typing import Any

from google.genai import Client, types
from pydantic import BaseModel, Field

### Task 1

In [5]:
class StepStatus(str, Enum):
    PENDING = "pending"
    RUNNING = "running"
    COMPLETED = "completed"
    FAILED = "failed"
    SKIPPED = "skipped"


class PlanStep(BaseModel):
    id: str = Field(description="unique id for each plan step")
    description: str = Field(description="Description for plan step")
    tool_name_hint: str | None = Field(default=None, description="Tool hint for this step")
    depends_on: list[str] = Field(
        default_factory=list, description="IDs of previous dependent steps"
    )
    success_criteria: str = Field(description="Criteria to mark success")
    status: StepStatus = Field(default=StepStatus.PENDING, description="Status of this step")
    result_summary: str | None = Field(default=None, description="Summary of this step result")
    attempts: int = Field(default=0, description="Attempts to complete this step")


class Plan(BaseModel):
    goal: str = Field(description="Goal for the agent")
    steps: list[PlanStep] = Field(description="Steps to complete goal")
    version: int = Field(default=1, description="Version of this plan")
    assumptions: list[str] = Field(default_factory=list, description="Assumptions for this goal")

### Task 2

In [6]:
def detect_cycle(steps: list[PlanStep]) -> bool:
    """Detect cycles in plan step dependencies using Kahn's algorithm.

    Returns:
        True if a cycle is detected, False if the plan is a valid DAG.
    """
    in_degree: dict[str, int] = {s.id: 0 for s in steps}
    adj: dict[str, list[str]] = {s.id: [] for s in steps}

    for step in steps:
        for dep in step.depends_on or []:
            if dep == step.id:
                return True  # Immediate self-dependency cycle
            if dep in adj:
                adj[dep].append(step.id)
                in_degree[step.id] += 1

    # Queue of nodes with zero incoming dependencies
    queue = [node_id for node_id, deg in in_degree.items() if deg == 0]
    visited_count = 0

    while queue:
        curr = queue.pop(0)
        visited_count += 1
        for neighbor in adj[curr]:
            in_degree[neighbor] -= 1
            if in_degree[neighbor] == 0:
                queue.append(neighbor)

    # If visited nodes count < total steps, a cycle exists
    return visited_count < len(steps)

### Task 3

In [ ]:
def validate_plan(plan: Plan) -> list[str]:
    """Validate a plan and return specific, model-readable error messages.

    Returns an empty list [] if the plan satisfies all DAG and metadata constraints.
    """
    errors: list[str] = []
    if not plan.steps:
        return ["Plan contains no steps. At least one step is required."]

    step_ids: set[str] = set()
    for step in plan.steps:
        if not step.id.strip():
            errors.append("Found step with empty ID.")
        elif step.id in step_ids:
            errors.append(f"Duplicate step ID '{step.id}'. Each step ID must be unique.")
        else:
            step_ids.add(step.id)

        if not step.description.strip():
            errors.append(f"Step '{step.id}' is missing a description.")
        if not step.success_criteria.strip():
            errors.append(f"Step '{step.id}' is missing success_criteria.")

    has_root = False
    for step in plan.steps:
        if not step.depends_on:
            has_root = True
        for dep in step.depends_on:
            if dep == step.id:
                errors.append(f"Step '{step.id}' depends on itself.")
            elif dep not in step_ids:
                errors.append(f"Step '{step.id}' depends on unknown step '{dep}'.")

    if not has_root and plan.steps:
        errors.append("Plan has no entry point (at least one step must have depends_on=[]).")

    # Check for cycles using Kahn's algorithm from Task 2 (only if IDs are unique)
    if len(step_ids) == len(plan.steps) and detect_cycle(plan.steps):
        errors.append("Circular dependency detected in plan steps.")

    return errors

### Task 4

In [7]:
def get_ready_steps(plan: Plan) -> list[PlanStep]:
    """Return steps that are ready to execute:

    status is PENDING and all dependencies are COMPLETED.
    """
    completed_ids = {step.id for step in plan.steps if step.status == StepStatus.COMPLETED}

    return [
        step
        for step in plan.steps
        if step.status == StepStatus.PENDING
        and all(dep in completed_ids for dep in step.depends_on)
    ]

### Task 5

In [10]:
async def execute_plan(
    plan: Plan,
    executor: Callable[[PlanStep], Awaitable[Any]],
    max_concurrency: int = 2,
) -> Plan:
    """Async scheduler that executes ready steps concurrently under a semaphore concurrency cap."""
    sem = asyncio.Semaphore(max_concurrency)
    running_tasks: dict[asyncio.Task, PlanStep] = {}

    async def run_step(step: PlanStep):
        async with sem:
            try:
                await executor(step)
                step.status = StepStatus.COMPLETED
            except Exception:
                step.status = StepStatus.FAILED
                raise

    while True:
        # 1. Schedule any newly ready steps
        for step in get_ready_steps(plan):
            step.status = StepStatus.IN_PROGRESS  # Claim step to prevent duplicate scheduling
            task = asyncio.create_task(run_step(step))
            running_tasks[task] = step

        if not running_tasks:
            # No tasks active and no new ready steps -> execution complete or halted
            break

        # 2. Wait for at least one running step to finish
        done, _ = await asyncio.wait(running_tasks.keys(), return_when=asyncio.FIRST_COMPLETED)
        for task in done:
            step = running_tasks.pop(task)
            if task.exception():
                print(f"Step '{step.id}' failed with error: {task.exception()}")

    return plan

### Task 6

In [ ]:
class JudgeVerdict(BaseModel):
    passed: bool
    reason: str


def verify_step(
    step: PlanStep,
    result: Any,
    predicate: Callable[[Any], bool] | None = None,
    client: Client | None = None,
    model: str = "gemini-3.5-flash-lite",
) -> tuple[bool, str]:
    """Verify a step result via programmatic predicate if defined,

    falling back to a small-model LLM judge when none is defined.
    """
    # 1. Programmatic Predicate Gate (fast, deterministic, zero-cost)
    if predicate is not None:
        try:
            passed = bool(predicate(result))
            reason = (
                "Programmatic predicate passed." if passed else "Programmatic predicate failed."
            )
            return passed, reason
        except Exception as exc:  # noqa
            return False, f"Programmatic predicate error: {exc}"

    # 2. Fallback: Small-Model Judge for semantic validation
    llm_client = client or Client()
    prompt = f"""Evaluate whether the step's result satisfies its success criteria.

Step Goal: {step.description}
Success Criteria: {step.success_criteria}

Result Deliverable:
{result}
"""
    response = llm_client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=JudgeVerdict,
        ),
    )
    verdict = JudgeVerdict.model_validate_json(response.text)
    return verdict.passed, verdict.reason

### Task 7

In [ ]:
class RevisedRemainder(BaseModel):
    revised_steps: list[PlanStep]


def replan(
    plan: Plan,
    failed_step_id: str,
    failure_reason: str,
    client: Client | None = None,
    model: str = "gemini-3.5-flash-lite",
) -> Plan:
    """Takes a failed plan, preserves completed steps, and requests a revised

    plan for the remainder incorporating the failure feedback.
    """
    # 1. Retain already completed steps
    completed_steps = [s for s in plan.steps if s.status == StepStatus.COMPLETED]
    completed_ids = [s.id for s in completed_steps]

    # 2. Query model to plan the remainder avoiding the failure
    llm_client = client or Client()
    prompt = f"""You are an expert replanning agent.
An execution plan failed during step '{failed_step_id}'.

Goal: {plan.goal}

Completed Steps (ALREADY DONE - DO NOT REPEAT):
{chr(10).join(f"- [{s.id}] {s.description}" for s in completed_steps) if completed_steps else "None"}

Failed Step:
- ID: {failed_step_id}
- Failure Reason: {failure_reason}

Instructions:
1. Generate revised replacement steps for the remainder to achieve the goal.
2. Fix, alter, or work around the failure in '{failed_step_id}'.
3. Do not re-execute completed steps {completed_ids}; connect new steps to them via depends_on.
4. Return only the revised remaining steps.
"""

    response = llm_client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=RevisedRemainder,
        ),
    )

    remainder = RevisedRemainder.model_validate_json(response.text)

    # 3. Assemble new plan: completed steps + revised remainder, increment version
    return Plan(
        goal=plan.goal,
        steps=completed_steps + remainder.revised_steps,
        version=plan.version + 1,
        assumptions=getattr(plan, "assumptions", []),
    )

### Task 8

In [14]:
DEFAULT_GATED_TOOLS = {
    "deploy_prod",
    "delete_database",
    "execute_migration",
    "send_payment",
    "refund_customer",
    "bash_exec",
}


def render_plan(
    plan: Plan,
    gated_tools: set[str] | None = None,
) -> str:
    """Renders a plan as human-readable text, flagging steps that use approval-gated tools."""
    approval_tools = gated_tools if gated_tools is not None else DEFAULT_GATED_TOOLS
    lines = [
        f"📋 Plan (v{plan.version}): {plan.goal}",
        "=" * 60,
    ]

    for idx, step in enumerate(plan.steps, 1):
        tool = getattr(step, "tool_name_hint", None)
        is_gated = tool in approval_tools if tool else False

        gate_badge = " [⚠️ APPROVAL REQUIRED]" if is_gated else ""
        tool_info = f" [Tool: {tool}]" if tool else ""
        deps_info = (
            f" (Depends on: {', '.join(step.depends_on)})" if step.depends_on else " (Initial Step)"
        )

        lines.append(f"{idx}. [{step.id}] {step.description}{gate_badge}")
        lines.append(f"   Status: {step.status.value.upper()}{deps_info}{tool_info}")
        lines.append(f"   Criteria: {step.success_criteria}")
        lines.append("")

    return "\n".join(lines).strip()

### Task 9

In [15]:
def diff_plans(old_plan: Plan, new_plan: Plan) -> str:
    """Produces a human-readable audit diff between two plan versions,

    identifying added, removed, and modified steps.
    """
    lines = [
        f"🔍 Plan Audit Diff: v{old_plan.version} ➔ v{new_plan.version}",
        "=" * 55,
    ]

    if old_plan.goal != new_plan.goal:
        lines.append(f"  Goal Changed: '{old_plan.goal}' ➔ '{new_plan.goal}'")

    old_steps = {s.id: s for s in old_plan.steps}
    new_steps = {s.id: s for s in new_plan.steps}

    old_ids = set(old_steps.keys())
    new_ids = set(new_steps.keys())

    # 1. Added Steps
    for step_id in sorted(new_ids - old_ids):
        s = new_steps[step_id]
        lines.append(f"[+] Added Step '{step_id}': {s.description} (deps={s.depends_on})")

    # 2. Removed Steps
    for step_id in sorted(old_ids - new_ids):
        s = old_steps[step_id]
        lines.append(
            f"[-] Removed Step '{step_id}': {s.description} (was {s.status.value.upper()})"
        )

    # 3. Modified Steps (present in both)
    for step_id in [s_id for s_id in new_steps if s_id in old_steps]:
        old_s = old_steps[step_id]
        new_s = new_steps[step_id]

        changes = []
        if old_s.status != new_s.status:
            changes.append(f"status: {old_s.status.value.upper()} ➔ {new_s.status.value.upper()}")
        if old_s.description != new_s.description:
            changes.append(f"description: '{old_s.description}' ➔ '{new_s.description}'")
        if old_s.depends_on != new_s.depends_on:
            changes.append(f"dependencies: {old_s.depends_on} ➔ {new_s.depends_on}")
        if getattr(old_s, "tool_name_hint", None) != getattr(new_s, "tool_name_hint", None):
            changes.append(
                f"tool: '{getattr(old_s, 'tool_name_hint', None)}' ➔"
                f" '{getattr(new_s, 'tool_name_hint', None)}'"
            )
        if old_s.success_criteria != new_s.success_criteria:
            changes.append(f"criteria: '{old_s.success_criteria}' ➔ '{new_s.success_criteria}'")

        if changes:
            lines.append(f"[~] Modified Step '{step_id}':")
            for c in changes:
                lines.append(f"      • {c}")

    if len(lines) == 2:
        lines.append("  (No changes detected between plan versions)")

    return "\n".join(lines)

### Task 10

In [21]:
@dataclass
class GoalRunResult:
    goal: str
    is_valid: bool
    is_completed: bool
    replans: int
    cost_usd: float


@dataclass
class EvalSummary:
    total_goals: int
    validity_rate: float
    completion_rate: float
    mean_replans: float
    mean_cost_usd: float
    results: list[GoalRunResult] = field(default_factory=list)

    def print_summary(self):
        print("\n" + "=" * 65)
        print("📊 AGENT PLANNING EVALUATION REPORT")
        print("=" * 65)
        print(f"Total Goals Evaluated : {self.total_goals}")
        print(f"Validity Rate         : {self.validity_rate * 100:.1f}%")
        print(f"Completion Rate       : {self.completion_rate * 100:.1f}%")
        print(f"Mean Replans per Goal : {self.mean_replans:.2f}")
        print(f"Mean Cost per Goal    : ${self.mean_cost_usd:.4f}")
        print("-" * 65)
        print(f"{'Goal':<30} | {'Valid':<5} | {'Done':<5} | {'Replans':<7} | {{'Cost':<7}}")
        print("-" * 65)
        for r in self.results:
            short_goal = r.goal[:27] + "..." if len(r.goal) > 30 else r.goal
            print(
                f"{short_goal:<30} | {'Yes' if r.is_valid else 'No':<5} |"
                f" {'Yes' if r.is_completed else 'No':<5} | {r.replans:<7} |"
                f" ${r.cost_usd:<.4f}"
            )
        print("=" * 65)


def evaluate_planning_agent(
    goals: list[str],
    run_goal_fn: Callable[[str], GoalRunResult],
) -> EvalSummary:
    """Evaluates agent planning performance over a set of test goals, computing

    validity rate, completion rate, mean replans, and mean cost per goal.
    """
    results = [run_goal_fn(g) for g in goals]
    n = len(results)
    if n == 0:
        return EvalSummary(0, 0.0, 0.0, 0.0, 0.0, [])

    valid_count = sum(1 for r in results if r.is_valid)
    completed_count = sum(1 for r in results if r.is_completed)
    total_replans = sum(r.replans for r in results)
    total_cost = sum(r.cost_usd for r in results)

    summary = EvalSummary(
        total_goals=n,
        validity_rate=valid_count / n,
        completion_rate=completed_count / n,
        mean_replans=total_replans / n,
        mean_cost_usd=total_cost / n,
        results=results,
    )
    summary.print_summary()
    return summary